# 05 · Model Interpretation and Business Use

This notebook compares the two decision points and translates model scores into a simple prioritization view.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

METRIC_DIR = Path('../outputs/model_metrics')
INTERP_DIR = Path('../outputs/interpretation')
PRED_DIR = Path('../outputs/predictions')

comparison = pd.read_csv(METRIC_DIR / 'placement_vs_delivery.csv')
importance = pd.read_csv(INTERP_DIR / 'catboost_feature_importance.csv')
comparison.round(3)

The placement model keeps more recall because it casts a wider net. The delivery model is more precise and ranks negative-review cases much better, which is reflected in both ROC-AUC and PR-AUC.

In [ ]:
importance.head(12)

The strongest delivery-stage features are mostly related to fulfillment timing and order complexity. These are predictive associations. They should not be read as causal estimates of what would happen if one variable were changed.

## Capture rate and lift

For an operational team, ranking quality is often easier to use than a single classification threshold. I therefore check how many negative reviews are found inside the highest-risk 5%, 10% and 20% of orders.

In [ ]:
def capture_table(stage, y, prob):
    y = np.asarray(y)
    prob = np.asarray(prob)
    base_rate = y.mean()
    order = np.argsort(-prob)
    y_sorted = y[order]

    rows = []
    for share in [0.05, 0.10, 0.20]:
        n = max(1, int(np.ceil(len(y) * share)))
        selected = y_sorted[:n]
        rows.append({
            'stage': stage,
            'top_share': share,
            'orders': n,
            'negative_reviews_captured': selected.sum() / y.sum(),
            'segment_negative_review_rate': selected.mean(),
            'lift_vs_average': selected.mean() / base_rate
        })
    return rows

placement = pd.read_csv(PRED_DIR / 'placement_predictions.csv')
delivery = pd.read_csv(PRED_DIR / 'delivery_predictions.csv')

lift = pd.DataFrame(
    capture_table(
        'At Placement',
        placement['actual_review_bad'],
        placement['placement_probability']
    )
    +
    capture_table(
        'At Delivery',
        delivery['actual_review_bad'],
        delivery['delivery_probability']
    )
)

lift.to_csv(METRIC_DIR / 'lift_capture.csv', index=False)
lift.round(3)

The delivery model is especially useful for prioritization. In the current test set, the top **10%** of delivery-risk scores contain about **43.6%** of all negative reviews, with a negative-review rate of about **55.9%**. That is roughly **4.4 times** the overall rate.

This does not establish the value of contacting those customers. It shows that the score concentrates risk well enough to support a targeted service-recovery experiment.